In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

In [2]:
train_data = datasets.PCAM(
    root='data',
    split='train',
    download=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [3]:
train_data_sub = Subset(train_data, range(6400))
len(train_data_sub)

6400

In [4]:
train_data_sub[0][0].shape, train_data_sub[0][1]

(torch.Size([3, 96, 96]), 0)

In [5]:
batch_size = 64
data_t = DataLoader(train_data_sub, batch_size=batch_size, shuffle=True)

In [6]:
for X, y in data_t:
    print(X.shape, y.shape, y.dtype)
    break

torch.Size([64, 3, 96, 96]) torch.Size([64]) torch.int64


In [7]:
test_data = datasets.PCAM(root='data', split='test', download=False,
                          transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]))

test_subset = Subset(test_data, range(128))

test_dl = DataLoader(test_subset, batch_size=batch_size)

In [8]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
device

'mps'

In [9]:
class PCamNet(nn.Module):
    def __init__(self, chans=(3, 16, 32, 64), r=4):
        super().__init__()
        n = len(chans) - 1
        self.convs  = nn.ModuleList([nn.Conv2d(chans[i], chans[i+1], 3, padding=1) for i in range(n)])
        self.bns    = nn.ModuleList([nn.BatchNorm2d(c) for c in chans[1:]])
        self.se_fc1 = nn.ModuleList([nn.Linear(c, c // r) for c in chans[1:]])   # SE squeeze
        self.se_fc2 = nn.ModuleList([nn.Linear(c // r, c) for c in chans[1:]])   # SE excite
        self.pool    = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(chans[-1], 1)

    def forward(self, x):                                   # (B, 3, 96, 96)
        for conv, bn, f1, f2 in zip(self.convs, self.bns, self.se_fc1, self.se_fc2):
            x = F.relu(bn(conv(x)))
            s = x.mean(dim=(2, 3))                          # SE: (B, C)
            w = torch.sigmoid(f2(F.relu(f1(s))))
            x = self.pool(x * w[:, :, None, None])          # 96 -> 48 -> 24 -> 12
        x = x.mean(dim=(2, 3))                              # global avg pool -> (B, 64)
        return self.fc(self.dropout(x)).squeeze(1)          # (B,) logits


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        targets = targets.float()
        # the negative sign is already calculated inside the PyTorch F.binary_cross_entropy_with_logits function.
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = torch.exp(-bce)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

In [11]:
# ---- setup ----
epochs = 10
model     = PCamNet().to(device)
criterion = FocalLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, total_steps=epochs * len(data_t)
)

In [16]:
model

PCamNet(
  (convs): ModuleList(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (bns): ModuleList(
    (0): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (se_fc1): ModuleList(
    (0): Linear(in_features=16, out_features=4, bias=True)
    (1): Linear(in_features=32, out_features=8, bias=True)
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
  (se_fc2): ModuleList(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): Linear(in_features=8, out_features=32, bias=True)
    (2): Linear(in_features=16, out_features=64, bias=True)
  )
  (pool): MaxPool2d(kernel_size=2, s

In [ ]:
# ---- train / eval ----
def train_one_epoch(loader):
    model.train()
    for i, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()

        optimizer.step()
        scheduler.step()                      # per batch (OneCycleLR)
        
        if i % 20 == 0:
            print(f"  loss {loss.item():.4f} [{i}/{len(loader)}]")

In [ ]:
def evaluate(loader):
    model.eval()
    probs, ys, total = [], [], 0.0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            logits = model(X)
            total += criterion(logits, y).item()

            probs.append(torch.sigmoid(logits).cpu())
            ys.append(y.cpu())
            
    probs, ys = torch.cat(probs), torch.cat(ys)
    acc = ((probs > 0.5).long() == ys).float().mean().item()
    return total / len(loader), acc, roc_auc_score(ys, probs)


In [14]:
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train_one_epoch(data_t)
    val_loss, acc, auc = evaluate(test_dl)
    print(f"  lr {scheduler.get_last_lr()[0]:.5f} | val loss {val_loss:.4f} | acc {acc:.3f} | AUC {auc:.3f}")


Epoch 1/10
  loss 0.0904 [0/100]
  loss 0.0745 [20/100]
  loss 0.0603 [40/100]
  loss 0.0632 [60/100]
  loss 0.0573 [80/100]
  lr 0.00084 | val loss 0.1152 | acc 0.320 | AUC 0.750
Epoch 2/10
  loss 0.0521 [0/100]
  loss 0.0416 [20/100]
  loss 0.0471 [40/100]
  loss 0.0476 [60/100]
  loss 0.0589 [80/100]
  lr 0.00229 | val loss 0.0992 | acc 0.383 | AUC 0.847
Epoch 3/10
  loss 0.0428 [0/100]
  loss 0.0394 [20/100]
  loss 0.0419 [40/100]
  loss 0.0542 [60/100]
  loss 0.0539 [80/100]
  lr 0.00300 | val loss 0.1835 | acc 0.312 | AUC 0.722
Epoch 4/10
  loss 0.0613 [0/100]
  loss 0.0537 [20/100]
  loss 0.0338 [40/100]
  loss 0.0470 [60/100]
  loss 0.0598 [80/100]
  lr 0.00285 | val loss 0.0637 | acc 0.570 | AUC 0.862
Epoch 5/10
  loss 0.0493 [0/100]
  loss 0.0377 [20/100]
  loss 0.0523 [40/100]
  loss 0.0418 [60/100]
  loss 0.0469 [80/100]
  lr 0.00243 | val loss 0.0552 | acc 0.570 | AUC 0.870
Epoch 6/10
  loss 0.0372 [0/100]
  loss 0.0311 [20/100]
  loss 0.0364 [40/100]
  loss 0.0534 [60/100

In [15]:
torch.save(model.state_dict(), "pcamnet.pth")

“Layers must be registered as attributes or placed in ModuleList/Sequential, or PyTorch won’t track their parameters.”

### Mathematical Formula

The standard cross-entropy (CE) loss for a true class probability $p_t$ is defined as:

$$ \text{CE}(p_t) = -\log(p_t) $$

Focal Loss introduces a modulating factor $(1 - p_t)^\gamma$ and a balancing weight $\alpha_t$ to the standard equation:

$$ \text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t) $$

Where:
* **$p_t$ (Probability of Ground Truth Class):** Defined as $p$ if the ground truth label is 1, and $1 - p$ otherwise (where $p$ is the model's estimated probability for the class).
* **$\gamma$ (Focusing Parameter):** Smoothly adjusts the rate at which easy examples are down-weighted. When $\gamma = 0$, Focal Loss becomes equivalent to standard cross-entropy.
* **$\alpha_t$ (Balancing Parameter):** Addressed via a conditional assignment based on the target class label:

$$ \alpha_t = \begin{cases} \alpha & \text{if target} = 1 \\ 1 - \alpha & \text{if target} = 0 \end{cases} $$


This alpha offsets the class imbalance of the number of examples. This alpha is set inversely proportional to the number of examples for a particular class or is learned through cross-validation.

This term is used to focus on the hard examples. The hard and easy examples are defined by the probability of the correct classification, $p_t$. If $p_t$ is low, then we are dealing with a hard sample. In this case, the modulating factor, $(1-p_t)^{\gamma}$ remains near 1 and is unaffected. If $p_t$ is high then the modulating factor vanishes to 0. This gives lower weight to the easy samples.